In [3]:
import pandas as pd
DATA_DIR= "../data"
import os
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
import healpy as hp
import fitsio
import matplotlib.pyplot as plt
from lsst.daf.butler import Butler

In [4]:
but = Butler("/global/cfs/cdirs/lsst/production/gen3/rubin/decam/repo/")

In [5]:
myDays = {"g":[],
"r":[],
"i":[],
"z":[],
         "all":[]}
for ref in but.query_all_datasets("DECam/raw/all",where="detector=3 AND instrument='DECam'"):
    myDays[ref.dataId['band']].append(ref.dataId['day_obs'])
    myDays["all"].append(ref.dataId['day_obs'])
for b in myDays.keys():
    myDays[b] = np.unique(myDays[b])

KeyError: 'opaque'

In [10]:
bias_days = np.loadtxt(os.path.join(DATA_DIR,"good_bias.txt"),dtype=int)
flat_days = np.loadtxt(os.path.join(DATA_DIR,"good_flat.txt"),dtype=int)

In [11]:
flatSpan = {"g":[],
"r":[],
"i":[],
"z":[]}

In [8]:
for b in flatSpan.keys():
    for band_day in myDays[b]:
        flatSpan[b].append(flat_days[band_day>flat_days][-1])
for b in flatSpan.keys():
    flatSpan[b] = np.unique(flatSpan[b])

In [9]:
for b in flatSpan.keys():
    
    with open(os.path.join(DATA_DIR,f"pilot_flat_{b}.txt"),"w+") as f:
        for x in flatSpan[b]:
            f.write(str(x))
            f.write("\n")

In [67]:
biasSpan = []
for band_day in myDays["all"]:
    biasSpan.append(bias_days[band_day>bias_days][-1])
biasSpan = np.unique(biasSpan)

In [71]:
with open(os.path.join(DATA_DIR,"pilot_bias.txt"),"w+") as f:
    for x in biasSpan:
        f.write(str(x))
        f.write("\n")

## Get the exposure numbers for the calibs

In [20]:
import requests
import os
from tqdm import tqdm

natroot = "https://astroarchive.noirlab.edu"
adsurl = f"{natroot}/api/adv_search"
DATA_DIR = "/global/homes/s/seanmacb/decam_template_tools/endurance/data"

#### Biases

In [23]:
def formatter(i):
    i=str(i)
    return f"{i[:4]}-{i[4:6]}-{i[6:]}"

In [28]:
year_month_date = []
for y in np.arange(2019,2026):
    for m in np.arange(1,13):
        if len(str(m))==1:
            m = f"0{m}"
        year_month_date.append(f"{y}{m}10")
year_month_date.append(20201011)
year_month_date.append(20201020)
year_month_date.append(20201120)
year_month_date.append(20201220)

In [29]:
for x in year_month_date:
    jj = {
        "outfields": [
            "md5sum",
            "archive_filename",
            "instrument",
            "proc_type",
            "obs_type",
        ],
        "search": [
            ["instrument", "decam"],
            ["proc_type", "raw"],
            # ["obs_type", "dome flat"],
            ["obs_type", "zero"],
            ["caldat", formatter(x),formatter(x)],  # The API natively understands this as a range
        ]
    }
    
    response = requests.post(f'{adsurl}/find/?limit=1000', json=jj)
    response = pd.DataFrame(response.json()[1:])
    if len(response)>0:
        print(f"{len(response)} bias exposures found for day {formatter(x)}.")
        with open(os.path.join(DATA_DIR,"calib_bias_md5s.txt"),"a") as f:
            for x in response['md5sum']:
                f.write(x)
                f.write("\n")
    else:
        print(f"No bias exposures found for day {formatter(x)}. Continuing.")

22 bias exposures found for day 2019-01-10.
22 bias exposures found for day 2019-02-10.
22 bias exposures found for day 2019-03-10.
23 bias exposures found for day 2019-04-10.
22 bias exposures found for day 2019-05-10.
No bias exposures found for day 2019-06-10. Continuing.
22 bias exposures found for day 2019-07-10.
22 bias exposures found for day 2019-08-10.
22 bias exposures found for day 2019-09-10.
No bias exposures found for day 2019-10-10. Continuing.
No bias exposures found for day 2019-11-10. Continuing.
22 bias exposures found for day 2019-12-10.
No bias exposures found for day 2020-01-10. Continuing.
22 bias exposures found for day 2020-02-10.
No bias exposures found for day 2020-03-10. Continuing.
No bias exposures found for day 2020-04-10. Continuing.
No bias exposures found for day 2020-05-10. Continuing.
No bias exposures found for day 2020-06-10. Continuing.
No bias exposures found for day 2020-07-10. Continuing.
No bias exposures found for day 2020-08-10. Continuing.


#### Flats

In [30]:
filters = {"y":'Y DECam c0005 10095.0 1130.0', 
           "g":'g DECam SDSS c0001 4720.0 1520.0',
           "i":'i DECam SDSS c0003 7835.0 1470.0',
           "r":'r DECam SDSS c0002 6415.0 1480.0', 
           "u":'u DECam c0006 3500.0 1000.0',
           "z":'z DECam SDSS c0004 9260.0 1520.0'
          }

In [37]:
for b in list("griz"):
    print(f"Band {b}.")
    print(10*"=====")

    for d in year_month_date:
        jj = {
            "outfields": [
                "md5sum",
                "archive_filename",
                "instrument",
                "proc_type",
                "obs_type",
                "FILTER"
            ],
            "search": [
                ["instrument", "decam"],
                ["proc_type", "raw"],
                ["obs_type", "dome flat"],
                ["FILTER", filters[b]],
                ["caldat", formatter(d),formatter(d)],  # The API natively understands this as a range
            ]
        }
        
        try:
            response = requests.post(f'{adsurl}/find/?limit=1000', json=jj)
            response = pd.DataFrame(response.json()[1:])
            if len(response)>0:
                print(f"{len(response)} flat exposures found for day {d}.")
                with open(os.path.join(DATA_DIR,f"{b}_calib_flat_md5s.txt"),"a") as f:
                    for x in response['md5sum']:
                        f.write(x)
                        f.write("\n")
            else:
                print(f"No flat exposures found for day {formatter(x)}. Continuing.")
        except Exception as e:
            print(f"Exception: {e}")
            continue

Band g.
11 flat exposures found for day 20190110.
11 flat exposures found for day 20190210.
11 flat exposures found for day 20190310.
11 flat exposures found for day 20190410.
11 flat exposures found for day 20190510.
No flat exposures found for day 510d-76-b894315ee0a1741ac6ac1bdb59. Continuing.
11 flat exposures found for day 20190710.
11 flat exposures found for day 20190810.
11 flat exposures found for day 20190910.
No flat exposures found for day fda9-a5-b2961faa760137cae4bf6e2b01. Continuing.
No flat exposures found for day fda9-a5-b2961faa760137cae4bf6e2b01. Continuing.
11 flat exposures found for day 20191210.
No flat exposures found for day e24c-04-b36de841304628a917c2f51c33. Continuing.
11 flat exposures found for day 20200210.
No flat exposures found for day 5511-ea-348a6646e8c6b5783ab4528d90. Continuing.
No flat exposures found for day 5511-ea-348a6646e8c6b5783ab4528d90. Continuing.
No flat exposures found for day 5511-ea-348a6646e8c6b5783ab4528d90. Continuing.
No flat expo